# Figure3_rank8_program_temporal_spatial_specificity

In [1]:

from pathlib import Path
import os, re, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import hypergeom
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset, random_split
    HAS_TORCH = True
except Exception as e:
    HAS_TORCH = False
    print('Torch unavailable:', e)

try:
    from tensorly.decomposition import parafac
    from tensorly.cp_tensor import cp_to_tensor
    HAS_TENSORLY = True
except Exception as e:
    HAS_TENSORLY = False
    print('Tensorly unavailable:', e)

project_dir = Path('/Users/sidaye/Documents/python/ST_MultiCAST')
input_dir = project_dir / 'Input'
base_output_dir = project_dir / 'Output'
model_comparison_dir = base_output_dir / 'model comparison'
ai_output_dir = base_output_dir / 'AI_spatiotemporal_models_python'
Spacepoints = ['st','SI1','SI2','SI3','SI4','SI5','SI6','SI7','SI8','SI9','ce','co']
Full_Timepoints = ['1h','3h','6h','12h','24h']
feature_order = [f'{t}_{s}' for t in Full_Timepoints for s in Spacepoints]

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['font.family'] = 'DejaVu Sans'

def save_pdf(fig, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches='tight', transparent=True)
    plt.close(fig)

def build_data():
    df = pd.read_csv(input_dir / 'Spatial_temporal_MultiSCAST_FC_final_capping.csv')
    df['Gene'] = df['Gene'].astype(str)
    df['Time'] = df['Time'].astype(str)
    df['Space'] = df['Space'].astype(str)
    df = df[df['Time'].isin(Full_Timepoints) & df['Space'].isin(Spacepoints)].copy()
    df['Feature'] = df['Time'] + '_' + df['Space']
    df = df.groupby(['Gene','Time','Space','Feature'], as_index=False).agg(logFC=('logFC','mean'))
    wide = df.pivot_table(index='Gene', columns='Feature', values='logFC', aggfunc='mean')
    wide = wide[[f for f in feature_order if f in wide.columns]].dropna(axis=0, how='any')
    X_raw_df = wide.copy()
    X_raw = X_raw_df.values.astype(float)
    row_mean = X_raw.mean(axis=1, keepdims=True)
    row_std = X_raw.std(axis=1, keepdims=True)
    row_std[row_std == 0] = 1.0
    X_scaled = np.nan_to_num((X_raw - row_mean) / row_std)
    X_scaled_df = pd.DataFrame(X_scaled, index=X_raw_df.index.astype(str), columns=X_raw_df.columns)
    return X_raw_df, X_scaled_df, row_mean, row_std

def vector_to_landscape(vector, columns=None):
    if columns is None:
        columns = feature_order
    s = pd.Series(np.asarray(vector, dtype=float), index=columns)
    return s.reindex(feature_order).values.reshape(len(Full_Timepoints), len(Spacepoints))

def load_category_table():
    cat = pd.read_excel(input_dir / 'putative_driver_gene_categories_12class.xlsx', sheet_name=0)
    cat['locus_ID'] = cat['locus_ID'].astype(str)
    col = 'Putative_driver_category'
    cat_map = cat.set_index('locus_ID')[col].dropna().to_dict()
    cats = sorted(pd.Series(cat_map).dropna().unique().tolist())
    palette = plt.cm.tab20(np.linspace(0, 1, max(20, len(cats))))
    color_map = {c: palette[i] for i, c in enumerate(cats)}
    default = '#2f6db3'
    return cat_map, color_map, default

def load_annotation():
    ann = pd.read_csv(input_dir / 'new_annotations_with_uniprot_names.csv')
    ann['locus_ID'] = ann['locus_ID'].astype(str)
    display_cols = ['gene_name','uniprot_gene_name','gene_name_old','KEGG_VC_number']
    def display(row):
        for c in display_cols:
            v = row.get(c, np.nan)
            if pd.notna(v) and str(v).strip() and str(v).lower() != 'nan':
                return str(v)
        return str(row['locus_ID'])
    ann['Gene_display'] = ann.apply(display, axis=1)
    text_cols = [c for c in ann.columns if c != 'locus_ID']
    ann['Annotation_text'] = ann[text_cols].astype(str).replace('nan','', regex=False).agg(' | '.join, axis=1)
    return ann

def phase_for_time(t):
    return {'1h':'Early','3h':'Early','6h':'Middle','12h':'Middle','24h':'Late'}.get(t, '')

def niche_for_space(s):
    if s == 'st': return 'stomach'
    if str(s).startswith('SI'): return 'small_intestine'
    if s == 'ce': return 'cecum'
    if s == 'co': return 'colon'
    return s

outdir = base_output_dir / 'Program_activation_specificity3'
outdir.mkdir(parents=True, exist_ok=True)

# Use rank8 program landscapes from Figure2. If this file was generated from a best-rank
# source, Requested_rank marks that we are showing the first 8 programs as rank8 panels.
land_path = base_output_dir / 'program_landscapes2' / 'Figure2_rank8_program_landscapes_long.csv'
if not land_path.exists():
    land_path = model_comparison_dir / 'PCA_CP_VAE_CPVAE_rank8_program_landscapes_long.csv'
land = pd.read_csv(land_path)
land['Program_number'] = land['Program'].str.extract(r'(\d+)').astype(int)
rank8 = land[land['Program_number'].between(1, 8)].copy()
rank8['Model'] = rank8['Model'].replace({'weighted CPVAE': 'weighted_CPVAE'})
model_order = ['PCA', 'CP', 'VAE', 'weighted_CPVAE']
model_titles = {'PCA': 'PCA', 'CP': 'CP', 'VAE': 'VAE', 'weighted_CPVAE': 'weighted CPVAE'}
rank8 = rank8[rank8['Model'].isin(model_order)].copy()
rank8['Program_label'] = rank8['Model'].map(model_titles) + ' ' + rank8['Program_number'].astype(str)
rank8['Abs_value'] = rank8['Value'].abs()

# Summaries: strength = mean absolute decoded program loading across the other axis.
temporal = (
    rank8.groupby(['Model', 'Program', 'Program_number', 'Program_label', 'Time'], as_index=False)
    .agg(Strength=('Abs_value', 'mean'), Signed_mean=('Value', 'mean'))
)
spatial = (
    rank8.groupby(['Model', 'Program', 'Program_number', 'Program_label', 'Space'], as_index=False)
    .agg(Strength=('Abs_value', 'mean'), Signed_mean=('Value', 'mean'))
)

def add_relative_strength(df, axis_col):
    out = df.copy()
    denom = out.groupby(['Model', 'Program_number'])['Strength'].transform('max').replace(0, np.nan)
    out['Relative_strength'] = (out['Strength'] / denom).fillna(0)
    return out

temporal = add_relative_strength(temporal, 'Time')
spatial = add_relative_strength(spatial, 'Space')
temporal.to_csv(outdir / 'Figure3_rank8_four_model_temporal_activation_strength.csv', index=False)
spatial.to_csv(outdir / 'Figure3_rank8_four_model_spatial_specificity_strength.csv', index=False)
rank8.to_csv(outdir / 'Figure3_rank8_four_model_program_landscapes_long.csv', index=False)

colors = plt.cm.tab10(np.linspace(0, 1, 8))

# Temporal line plots: one panel per model, each line is one rank8 program.
fig, axes = plt.subplots(2, 2, figsize=(12.2, 7.4), sharex=True, sharey=True)
axes = axes.ravel()
for ax, model in zip(axes, model_order):
    sub = temporal[temporal['Model'].eq(model)].copy()
    for pnum in range(1, 9):
        line = sub[sub['Program_number'].eq(pnum)].set_index('Time').reindex(Full_Timepoints)
        ax.plot(Full_Timepoints, line['Relative_strength'], marker='o', linewidth=1.4, markersize=3.5, color=colors[pnum-1], label=f'Program {pnum}')
    ax.set_title(model_titles[model], fontsize=10, pad=8)
    ax.set_ylim(-0.03, 1.08)
    ax.set_ylabel('Relative temporal strength')
    ax.grid(True, axis='y', linewidth=0.4, alpha=0.35)
    ax.tick_params(axis='x', labelrotation=0)
axes[-1].legend(title='Rank8 program', bbox_to_anchor=(1.03, 0.5), loc='center left', frameon=False, fontsize=7, title_fontsize=8)
fig.suptitle('Rank8 program temporal activation across four models', fontsize=12.5, y=0.965)
fig.subplots_adjust(top=0.88, bottom=0.10, left=0.08, right=0.84, hspace=0.38, wspace=0.24)
save_pdf(fig, outdir / 'Figure3_rank8_four_model_program_temporal_activation_lines.pdf')

# Spatial line plots: one panel per model, each line is one rank8 program.
fig, axes = plt.subplots(2, 2, figsize=(13.4, 7.6), sharex=True, sharey=True)
axes = axes.ravel()
for ax, model in zip(axes, model_order):
    sub = spatial[spatial['Model'].eq(model)].copy()
    for pnum in range(1, 9):
        line = sub[sub['Program_number'].eq(pnum)].set_index('Space').reindex(Spacepoints)
        ax.plot(Spacepoints, line['Relative_strength'], marker='o', linewidth=1.35, markersize=3.3, color=colors[pnum-1], label=f'Program {pnum}')
    ax.set_title(model_titles[model], fontsize=10, pad=8)
    ax.set_ylim(-0.03, 1.08)
    ax.set_ylabel('Relative spatial strength')
    ax.grid(True, axis='y', linewidth=0.4, alpha=0.35)
    ax.tick_params(axis='x', labelrotation=45)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment('right')
axes[-1].legend(title='Rank8 program', bbox_to_anchor=(1.03, 0.5), loc='center left', frameon=False, fontsize=7, title_fontsize=8)
fig.suptitle('Rank8 program spatial specificity across four models', fontsize=12.5, y=0.965)
fig.subplots_adjust(top=0.88, bottom=0.16, left=0.08, right=0.84, hspace=0.42, wspace=0.24)
save_pdf(fig, outdir / 'Figure3_rank8_four_model_program_spatial_specificity_lines.pdf')

# Heatmap helper: rows are model-programs, columns are original time/location labels.
def make_heatmap_table(df, value_col, column_col, column_order):
    blocks = []
    row_labels = []
    for model in model_order:
        sub = df[df['Model'].eq(model)].copy()
        for pnum in range(1, 9):
            row = sub[sub['Program_number'].eq(pnum)].set_index(column_col).reindex(column_order)[value_col]
            blocks.append(row.values.astype(float))
            row_labels.append(f"{model_titles[model]} P{pnum}")
    return np.vstack(blocks), row_labels

tmat, tlabels = make_heatmap_table(temporal, 'Relative_strength', 'Time', Full_Timepoints)
fig, ax = plt.subplots(figsize=(7.2, 10.2))
im = ax.imshow(tmat, aspect='auto', cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(range(len(Full_Timepoints)))
ax.set_xticklabels(Full_Timepoints, fontsize=9)
ax.set_yticks(range(len(tlabels)))
ax.set_yticklabels(tlabels, fontsize=6.5)
for y in [7.5, 15.5, 23.5]:
    ax.axhline(y, color='white', linewidth=1.0)
ax.set_title('Rank8 temporal activation strength', fontsize=12, pad=12)
ax.set_xlabel('Time')
ax.set_ylabel('Model / program')
cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.025)
cbar.set_label('Relative strength', fontsize=9)
fig.subplots_adjust(top=0.93, bottom=0.07, left=0.28, right=0.90)
save_pdf(fig, outdir / 'Figure3_rank8_four_model_temporal_activation_heatmap.pdf')

smat, slabels = make_heatmap_table(spatial, 'Relative_strength', 'Space', Spacepoints)
fig, ax = plt.subplots(figsize=(10.6, 10.2))
im = ax.imshow(smat, aspect='auto', cmap='magma', vmin=0, vmax=1)
ax.set_xticks(range(len(Spacepoints)))
ax.set_xticklabels(Spacepoints, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(slabels)))
ax.set_yticklabels(slabels, fontsize=6.5)
for y in [7.5, 15.5, 23.5]:
    ax.axhline(y, color='white', linewidth=1.0)
ax.set_title('Rank8 spatial specificity strength', fontsize=12, pad=12)
ax.set_xlabel('Location')
ax.set_ylabel('Model / program')
cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
cbar.set_label('Relative strength', fontsize=9)
fig.subplots_adjust(top=0.93, bottom=0.13, left=0.20, right=0.91)
save_pdf(fig, outdir / 'Figure3_rank8_four_model_spatial_specificity_heatmap.pdf')

print(outdir)


/Users/sidaye/Documents/python/ST_MultiCAST/Output/Program_activation_specificity3
